<a href="https://colab.research.google.com/github/PriscilaForcel/DOUTORADO/blob/main/BR_UNIVERSO_DESENVOLVIMENTO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Censo Demográfico 2022 - Banco Nacional Integrado por Setor Censitário (V2)

## Incorporação das Características Urbanísticas do Entorno dos Domicílios

**Versão:** 2.0.0 - preparação para publicação  
**Abrangência geográfica:** Brasil  
**Unidade territorial:** Setor censitário  
**Fonte dos dados:** Instituto Brasileiro de Geografia e Estatística (IBGE)  
**Ambiente de processamento:** Python / Google Colab

### 1. Apresentação

Este notebook documenta os procedimentos de ampliação do banco nacional integrado de agregados por setores censitários do Censo Demográfico 2022, mediante a incorporação do conjunto de Características Urbanísticas do Entorno dos Domicílios, disponibilizado pelo Instituto Brasileiro de Geografia e Estatística (IBGE).

A versão original (V1) reúne os conjuntos Básico, Características do Domicílio (Partes 1, 2 e 3), Demografia, Parentesco, Cor ou Raça e Alfabetização, totalizando 468.099 setores censitários e 1.356 colunas.

A presente versão (V2) acrescenta 35 variáveis relacionadas às características urbanísticas do entorno dos domicílios, identificadas pelo prefixo `ENT_`.

### 2. Procedimento metodológico

A integração utiliza o código do setor censitário (`CD_SETOR`) como chave territorial e a operação `LEFT JOIN`, preservando a totalidade dos registros existentes no banco original.

O processamento compreende a aquisição dos dados, a inspeção dos dicionários, a padronização das variáveis, a integração nacional, a validação dos resultados e a geração da documentação técnica.

As variáveis são preservadas como texto, sem conversão automática de valores numéricos ou de marcadores especiais.

### 3. Estrutura esperada da V2

| Característica | Resultado |
|---|---|
| Setores censitários | 468.099 |
| Colunas originais | 1.356 |
| Variáveis acrescentadas | 35 |
| Total de colunas | 1.391 |
| Setores com correspondência no Entorno | 340.965 |
| Setores sem correspondência no Entorno | 127.134 |

Os setores sem correspondência no conjunto Entorno são preservados com valores nulos nas variáveis acrescentadas. Essa ausência de correspondência não deve ser interpretada como ausência de infraestrutura urbana.

### 4. Reprodutibilidade

O notebook utiliza arquivos oficiais do IBGE e o banco consolidado V1, disponibilizado no Zenodo:

https://doi.org/10.5281/zenodo.23599214

Os caminhos de entrada e saída são configuráveis, permitindo executar os procedimentos sem depender da estrutura pessoal de diretórios utilizada durante o desenvolvimento.

### 5. Produtos gerados

- `UNIVERSO_2022_NACIONAL_V2.parquet`
- `DICIONARIO_COMPLETO_2022_V2.csv`
- `DICIONARIO_COMPLETO_2022_V2.xlsx`
- `RELATORIO_VALIDACAO_UNIVERSO_2022_V2.xlsx`
- `RELATORIO_TECNICO_UNIVERSO_2022_V2.md`

A validação compreende a correspondência territorial, a preservação das variáveis originais e a comparação das 35 novas variáveis com o arquivo de origem do IBGE.

**Nota:** este banco constitui uma integração produzida no âmbito de pesquisa acadêmica, a partir de dados oficiais do IBGE. Não representa uma base integrada publicada oficialmente pelo Instituto.


# 1. CONFIGURAÇÃO DO AMBIENTE DE PROCESSAMENTO

Objetivo:
Definir bibliotecas, conexão DuckDB e diretórios de trabalho.

Para utilizar o Google Drive, monte-o e altere PASTA_BASE.

A configuração pública utiliza o armazenamento do Colab.

In [ ]:

%pip install -q duckdb pandas pyarrow openpyxl xlsxwriter requests beautifulsoup4

from pathlib import Path
import duckdb
import pandas as pd
import requests
import zipfile
import io
import time
import hashlib

# Diretório configurável de trabalho
PASTA_BASE = Path("/content/IBGE_UNIVERSO")

# Exemplo alternativo para Google Drive:
#
# from google.colab import drive
# drive.mount("/content/drive")
# PASTA_BASE = Path("/content/drive/MyDrive/IBGE_UNIVERSO")

PASTA_ENTORNO = PASTA_BASE / "ENTORNO"
PASTA_RESULTADOS = PASTA_BASE / "RESULTADOS_2022"
PASTA_DICIONARIOS = PASTA_RESULTADOS / "DICIONARIO_2022"

for pasta in [
    PASTA_BASE,
    PASTA_ENTORNO,
    PASTA_RESULTADOS,
    PASTA_DICIONARIOS
]:
    pasta.mkdir(parents=True, exist_ok=True)

# Conexão DuckDB
con = duckdb.connect()
con.execute("SET threads = 2")
con.execute("SET memory_limit = '2GB'")
con.execute("SET temp_directory = '/content/duckdb_temp'")

# Controle de proteção dos arquivos produzidos
SOBRESCREVER_SAIDAS = False

print("Ambiente configurado.")
print("Diretório de trabalho:", PASTA_BASE)


# Inventário dos dados de Entorno

Esta etapa identifica os arquivos disponibilizados no diretório de entrada, incluindo os dicionários oficiais do IBGE e os arquivos nacionais de agregados por setores censitários. O inventário permite verificar quais fontes estão disponíveis antes do processamento.

In [ ]:
# 4. IDENTIFICAÇÃO DOS ARQUIVOS - ENTORNO

from google.colab import drive
from pathlib import Path
import pandas as pd

# Montar o Google Drive
drive.mount("/content/drive")

# Identificar os arquivos disponíveis no diretório ENTORNO
pasta_entorno = PASTA_ENTORNO

# Verificar existência
if not pasta_entorno.exists():
    raise FileNotFoundError(
        f"Pasta não encontrada: {pasta_entorno}"
    )

# Identificar os arquivos, inclusive em subpastas
arquivos = sorted(
    [p for p in pasta_entorno.rglob("*") if p.is_file()]
)

inventario = pd.DataFrame([
    {
        "Arquivo": p.name,
        "Extensão": p.suffix.lower(),
        "Tamanho (MB)": round(
            p.stat().st_size / 1024**2, 2
        ),
        "Subpasta": str(
            p.parent.relative_to(pasta_entorno)
        )
    }
    for p in arquivos
])

print("INVENTÁRIO -ENTORNO")
print("Total de arquivos:", len(inventario))

display(inventario)


# Fontes e dicionários do Entorno

Esta etapa inspeciona os dicionários de dados e identifica os arquivos nacionais publicados pelo IBGE. Para a ampliação da base, será utilizado exclusivamente o conjunto Entorno dos Domicílios, excluindo os conjuntos de faces e moradores.

In [ ]:
# 5. INSPEÇÃO DOS DICIONÁRIOS E ARQUIVOS DO ENTORNO

import zipfile
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# Pasta ENTORNO já identificada na célula anterior
arquivo_dicionario = (
    pasta_entorno / "dicionarios_de_dados_entorno.zip"
)

print("=== DICIONÁRIOS DISPONÍVEIS ===")

with zipfile.ZipFile(arquivo_dicionario) as z:
    for nome in z.namelist():
        if nome.lower().endswith((".xlsx", ".xls", ".csv")):
            print(nome)

            if nome.lower().endswith(".xlsx"):
                with z.open(nome) as f:
                    planilhas = pd.ExcelFile(f)
                    print("Abas:", planilhas.sheet_names)

print("\n=== ARQUIVOS NACIONAIS DO IBGE ===")

url_ibge = (
    "https://ftp.ibge.gov.br/Censos/"
    "Censo_Demografico_2022/"
    "Agregados_por_Setores_Censitarios_"
    "Caracteristicas_urbanisticas_do_entorno_dos_domicilios/"
    "Agregados_por_Setor_csv/"
)

resposta = requests.get(url_ibge, timeout=60)
resposta.raise_for_status()

soup = BeautifulSoup(resposta.text, "html.parser")

arquivos_ibge = []

for link in soup.find_all("a", href=True):
    nome = link.get("href")

    if nome.lower().endswith(".zip"):
        arquivos_ibge.append({
            "Arquivo": nome,
            "URL": urljoin(url_ibge, nome)
        })

display(pd.DataFrame(arquivos_ibge))


### 6. Aquisição dos agregados nacionais de Entorno dos Domicílios

Esta etapa identifica e obtém, diretamente do repositório oficial do Instituto Brasileiro de Geografia e Estatística (IBGE), o arquivo nacional de agregados por setores censitários das Características Urbanísticas do Entorno dos Domicílios, referente ao Censo Demográfico 2022.

O procedimento seleciona exclusivamente o conjunto associado aos domicílios, sem incorporar os arquivos relativos às faces de quadra ou aos moradores.

O arquivo é armazenado em seu formato ZIP original. Antes de realizar o download, o código verifica se já existe uma cópia íntegra no diretório configurado. Essa verificação evita transferências desnecessárias e preserva o arquivo de origem para as etapas posteriores de processamento e validação.

In [ ]:

# 6. DOWNLOAD NACIONAL -ENTORNO DOS DOMICÍLIOS


from urllib.parse import unquote
import requests
import zipfile

# 1. Localizar exclusivamente o conjunto de domicílios

arquivos_domicilios = [
    arq for arq in arquivos_ibge
    if "domicilios" in unquote(arq["Arquivo"]).lower()
    or "domicílios" in unquote(arq["Arquivo"]).lower()
]

if len(arquivos_domicilios) != 1:
    raise ValueError(
        f"Esperado 1 arquivo de domicílios; "
        f"encontrados {len(arquivos_domicilios)}."
    )

arquivo = arquivos_domicilios[0]
url_download = arquivo["URL"]

nome_arquivo = unquote(arquivo["Arquivo"])

destino = PASTA_ENTORNO / nome_arquivo
temporario = PASTA_ENTORNO / (nome_arquivo + ".part")

print("Arquivo selecionado:", nome_arquivo)
print("Destino:", destino)

# 2. Função de verificação da integridade do ZIP

def zip_integro(caminho):
    """
    Verifica se o arquivo existe, possui estrutura ZIP
    válida e se seus componentes passam na verificação CRC.
    """
    if not caminho.is_file():
        return False

    if not zipfile.is_zipfile(caminho):
        return False

    try:
        with zipfile.ZipFile(caminho) as z:
            return z.testzip() is None
    except (OSError, zipfile.BadZipFile):
        return False

# 3. Realizar download somente quando necessário

if zip_integro(destino):

    print("Arquivo ZIP íntegro já disponível.")
    print("Download dispensado.")

else:

    print("Iniciando download do arquivo nacional...")

    with requests.get(
        url_download,
        stream=True,
        timeout=(30, 120)
    ) as resposta:

        resposta.raise_for_status()

        with open(temporario, "wb") as f:
            for bloco in resposta.iter_content(
                chunk_size=1024 * 1024
            ):
                if bloco:
                    f.write(bloco)

    # Validar o arquivo antes de torná-lo definitivo
    if not zip_integro(temporario):
        temporario.unlink(missing_ok=True)
        raise ValueError(
            "O arquivo baixado não passou na "
            "verificação de integridade ZIP."
        )

    temporario.replace(destino)

    print("Download concluído e ZIP validado.")

# 4. Inspecionar os arquivos internos sem extração

with zipfile.ZipFile(destino) as z:

    print("\n=== CONTEÚDO DO ARQUIVO NACIONAL ===")

    for info in z.infolist():
        if not info.is_dir():
            print(
                f"Arquivo: {info.filename}\n"
                f"Tamanho descompactado: "
                f"{info.file_size / 1024**2:.2f} MB"
            )

print("\nEtapa 6 concluída.")


# Inspeção da estrutura das variáveis

São examinados o dicionário oficial e o cabeçalho do arquivo nacional para identificar os códigos das variáveis, suas descrições e a estrutura do identificador territorial. A leitura preliminar não altera os dados originais.

In [ ]:
# 7. INSPEÇÃO DO CONJUNTO ENTORNO -DOMICÍLIOS

import pandas as pd
import zipfile
import io

# --------------------------------------------------
# 1. LER O DICIONÁRIO
# --------------------------------------------------

arquivo_dicionario = (
    pasta_entorno / "dicionarios_de_dados_entorno.zip"
)

with zipfile.ZipFile(arquivo_dicionario) as z:
    nome_dic = next(
        nome for nome in z.namelist()
        if nome.endswith("dicionario_entorno_domicilios.xlsx")
    )

    with z.open(nome_dic) as f:
        conteudo = f.read()

dicionario_entorno = pd.read_excel(
    io.BytesIO(conteudo),
    sheet_name="dicionario_entorno_domicilios_v",
    header=None
)

print("=== DICIONÁRIO -ENTORNO DOMICÍLIOS ===")
print("Dimensões:", dicionario_entorno.shape)

display(dicionario_entorno.head(25))

# --------------------------------------------------
# 2. IDENTIFICAR O CSV NACIONAL
# --------------------------------------------------

arquivo_zip = (
    pasta_entorno /
    "Agregados_por_setores_entorno_domicílios_BR.zip"
)

with zipfile.ZipFile(arquivo_zip) as z:
    nome_csv = next(
        nome for nome in z.namelist()
        if nome.lower().endswith(".csv")
    )

    # Ler apenas o cabeçalho e cinco observações
    with z.open(nome_csv) as f:
        amostra_entorno = pd.read_csv(
            f,
            sep=";",
            encoding="utf-8-sig",
            dtype=str,
            nrows=5
        )

print("\n=== CSV NACIONAL -ENTORNO DOMICÍLIOS ===")
print("Colunas:", len(amostra_entorno.columns))

print("\nNomes das colunas:")
print(amostra_entorno.columns.tolist())

print("\nPrimeiras observações:")
display(amostra_entorno.iloc[:, :15])


# Preparação do conjunto Entorno

Os dados nacionais são carregados como texto. O identificador CD_setor é padronizado para CD_SETOR, e as 35 variáveis temáticas recebem o prefixo ENT_, evitando conflitos de nomenclatura com a base original. São verificadas a unicidade dos identificadores, a ausência de códigos nulos e a quantidade de registros.

In [ ]:

# 8. CARREGAMENTO E VALIDAÇÃO -ENTORNO DOS DOMICÍLIOS

import pandas as pd
import zipfile

# --------------------------------------------------
# 1. CARREGAR O CSV ORIGINAL DO IBGE
# --------------------------------------------------

with zipfile.ZipFile(arquivo_zip) as z:
    with z.open(nome_csv) as f:
        entorno = pd.read_csv(
            f,
            sep=";",
            encoding="utf-8-sig",
            dtype=str,
            low_memory=False
        )

# --------------------------------------------------
# 2. PADRONIZAR A CHAVE TERRITORIAL
# --------------------------------------------------

assert "CD_setor" in entorno.columns, (
    "A coluna CD_setor não foi encontrada no arquivo original."
)

entorno = entorno.rename(
    columns={"CD_setor": "CD_SETOR"}
)

# --------------------------------------------------
# 3. IDENTIFICAR E CONFERIR AS VARIÁVEIS TEMÁTICAS
# --------------------------------------------------

variaveis_entorno = [
    coluna for coluna in entorno.columns
    if coluna.startswith("V05")
]

variaveis_esperadas = [
    f"V050{i:02d}" for i in range(35)
]

assert variaveis_entorno == variaveis_esperadas, (
    "As variáveis de Entorno diferem das 35 "
    "variáveis esperadas (V05000 a V05034)."
)

assert len(entorno.columns) == 36, (
    "O arquivo deve conter CD_SETOR e 35 variáveis temáticas."
)

# --------------------------------------------------
# 4. VALIDAR OS IDENTIFICADORES TERRITORIAIS
# --------------------------------------------------

total_registros = len(entorno)
setores_unicos = entorno["CD_SETOR"].nunique()

duplicidades = entorno["CD_SETOR"].duplicated().sum()

setores_sem_codigo = (
    entorno["CD_SETOR"].isna() |
    entorno["CD_SETOR"].str.strip().eq("")
).sum()

print("=== VALIDAÇÃO DO ENTORNO -BRASIL ===")
print(f"Total de registros: {total_registros:,}")
print(f"Setores únicos: {setores_unicos:,}")
print(f"Duplicidades: {duplicidades:,}")
print(f"Setores sem código: {setores_sem_codigo:,}")
print(f"Variáveis temáticas: {len(variaveis_entorno)}")

# Conferir a estrutura da fonte utilizada na V2
assert total_registros == 340965, (
    "Quantidade de setores diferente da fonte "
    "utilizada na construção da V2."
)

assert setores_unicos == 340965, (
    "Quantidade inesperada de setores únicos."
)

assert duplicidades == 0, (
    "Existem setores duplicados no Entorno!"
)

assert setores_sem_codigo == 0, (
    "Existem setores sem identificador!"
)

# --------------------------------------------------
# 5. APLICAR O PREFIXO TEMÁTICO ENT_
# --------------------------------------------------

entorno = entorno.rename(
    columns={
        var: f"ENT_{var}"
        for var in variaveis_entorno
    }
)

# Conferir as colunas após a padronização
novas_colunas = [
    coluna for coluna in entorno.columns
    if coluna != "CD_SETOR"
]

assert novas_colunas == [
    f"ENT_{var}" for var in variaveis_esperadas
], "A padronização dos nomes não corresponde ao esperado."

# --------------------------------------------------
# 6. APRESENTAR O RESULTADO DA PREPARAÇÃO
# --------------------------------------------------

print("\nPADRONIZAÇÃO CONCLUÍDA!")
print("Total de setores:", len(entorno))
print("Variáveis com prefixo ENT_:", len(novas_colunas))
print("Chave territorial: CD_SETOR")

display(entorno.head(5).iloc[:, :10])


In [ ]:

# 9. OBTENÇÃO DA BASE NACIONAL V1 -ZENODO
#
# Objetivo:
# Disponibilizar o Parquet original para a integração
# com as variáveis de Entorno.
#
# Fonte: https://doi.org/10.5281/zenodo.23599214
#
# O arquivo original não será alterado.

ZENODO_ID_V1 = "23599214"

arquivo_v1 = (
    PASTA_BASE / "UNIVERSO_2022_NACIONAL.parquet"
)

# Consultar os metadados oficiais do depósito
url_api = (
    f"https://zenodo.org/api/records/{ZENODO_ID_V1}"
)

resposta = requests.get(url_api, timeout=60)
resposta.raise_for_status()
registro = resposta.json()

parquets = [
    arq for arq in registro["files"]
    if arq["key"] == "UNIVERSO_2022_NACIONAL.parquet"
]

assert len(parquets) == 1, (
    "Parquet original não identificado no Zenodo."
)

registro_parquet = parquets[0]
url_parquet = registro_parquet["links"]["self"]

tamanho_esperado = registro_parquet.get("size")
checksum = registro_parquet.get("checksum")

# Transferir somente quando necessário
if not arquivo_v1.exists():

    temporario = arquivo_v1.with_name(
        arquivo_v1.name + ".part"
    )

    print("Obtendo a versão original do Zenodo...")

    try:
        with requests.get(
            url_parquet,
            stream=True,
            timeout=(30, 180)
        ) as r:
            r.raise_for_status()

            with temporario.open("wb") as f:
                for bloco in r.iter_content(4 * 1024 * 1024):
                    if bloco:
                        f.write(bloco)

        if tamanho_esperado is not None:
            assert temporario.stat().st_size == int(
                tamanho_esperado
            )

        temporario.replace(arquivo_v1)

    except Exception:
        temporario.unlink(missing_ok=True)
        raise

# Verificar tamanho e integridade
if tamanho_esperado is not None:
    assert arquivo_v1.stat().st_size == int(tamanho_esperado)

if checksum and ":" in checksum:
    algoritmo, esperado = checksum.split(":", 1)
    algoritmo = algoritmo.lower()

    if algoritmo in ("md5", "sha256"):
        h = hashlib.new(algoritmo)
        with arquivo_v1.open("rb") as f:
            for bloco in iter(
                lambda: f.read(4 * 1024 * 1024), b""
            ):
                h.update(bloco)

        assert h.hexdigest().lower() == esperado.lower()

print("Base V1 disponível e integridade conferida.")
print("Arquivo:", arquivo_v1)


# Leitura local da versão original

O Parquet V1 é registrado como uma visualização no DuckDB. A operação permite realizar consultas e integrações sem carregar simultaneamente todas as colunas em um DataFrame Pandas. São conferidas as 468.099 linhas e as 1.356 colunas esperadas.

In [ ]:

# 10. REGISTRAR O BANCO LOCAL NO DUCKDB

caminho_v1_sql = str(arquivo_v1).replace("'", "''")

con.execute(f"""
    CREATE OR REPLACE VIEW universo_v1_local AS
    SELECT *
    FROM read_parquet('{caminho_v1_sql}')
""")

total = con.execute("""
    SELECT COUNT(*)
    FROM universo_v1_local
""").fetchone()[0]

colunas = con.execute("""
    DESCRIBE SELECT * FROM universo_v1_local
""").fetchdf()

assert total == 468099
assert len(colunas) == 1356

print("BASE NACIONAL V1 -LEITURA LOCAL")
print(f"Setores: {total:,}".replace(",", "."))
print(f"Colunas: {len(colunas):,}".replace(",", "."))
print("Conexão local estabelecida.")


# Correspondência territorial entre Universo e Entorno

Antes da integração, verifica-se a unicidade dos identificadores CD_SETOR na base original e a correspondência exata dos códigos presentes no Entorno. Essa etapa impede a multiplicação de registros e identifica eventuais códigos sem correspondência.

In [ ]:
# 11. VALIDAÇÃO TERRITORIAL -UNIVERSO V1 × ENTORNO

# Registrar o DataFrame de Entorno no DuckDB
con.register("entorno_dados", entorno)

# Conferir possíveis conflitos de nomes
colunas_v1 = set(
    con.execute(
        "DESCRIBE SELECT * FROM universo_v1_local"
    ).fetchdf()["column_name"]
)

novas_colunas = [
    c for c in entorno.columns if c != "CD_SETOR"
]

conflitos = sorted(colunas_v1.intersection(novas_colunas))

if conflitos:
    raise ValueError(
        f"Colunas já existentes na V1: {conflitos}"
    )

# Validar a chave da base original
integridade_v1 = con.execute("""
    SELECT
        COUNT(*) AS registros,
        COUNT(DISTINCT CD_SETOR) AS setores_unicos,
        COUNT(*) FILTER (
            WHERE CD_SETOR IS NULL
               OR TRIM(CD_SETOR) = ''
        ) AS codigos_ausentes
    FROM universo_v1_local
""").fetchdf()

display(integridade_v1)

assert integridade_v1.loc[0, "registros"] == 468099
assert integridade_v1.loc[0, "setores_unicos"] == 468099
assert integridade_v1.loc[0, "codigos_ausentes"] == 0

assert correspondencia.loc[0, "setores_entorno"] == 340965
assert correspondencia.loc[0, "correspondentes"] == 340965
assert correspondencia.loc[0, "sem_correspondencia"] == 0

# Comparar as chaves territoriais
correspondencia = con.execute("""
    SELECT
        COUNT(*) AS setores_entorno,
        COUNT(b.CD_SETOR) AS correspondentes,
        COUNT(*) FILTER (
            WHERE b.CD_SETOR IS NULL
        ) AS sem_correspondencia
    FROM entorno_dados e
    LEFT JOIN universo_v1_local b
        ON e.CD_SETOR = b.CD_SETOR
""").fetchdf()

display(correspondencia)

print("Variáveis novas sem conflito:", len(novas_colunas))
print("Validação de correspondência concluída.")


# Integração e exportação do banco nacional V2

A integração é executada por meio de LEFT JOIN, utilizando a versão original como referência territorial e CD_SETOR como chave única. São acrescentadas exclusivamente as 35 variáveis de Entorno. O novo banco é exportado em Apache Parquet com compressão ZSTD, sem modificar a V1.

In [ ]:
# 12. CONSTRUÇÃO DO BANCO NACIONAL -VERSÃO 2

from pathlib import Path

# Arquivo final da V2
arquivo_v2 = (
    PASTA_RESULTADOS /
    "UNIVERSO_2022_NACIONAL_V2.parquet"
)

# Arquivo intermediário de exportação
arquivo_v2_temp = (
    PASTA_RESULTADOS /
    "UNIVERSO_2022_NACIONAL_V2.parquet.tmp"
)

# Proteger resultados existentes
if arquivo_v2.exists() and not SOBRESCREVER_SAIDAS:
    raise FileExistsError(
        "A V2 já existe. A sobrescrita está desabilitada."
    )

# Evitar conflitos entre as colunas
assert len(novas_colunas) == 35

# Construir uma seleção explícita das novas variáveis
selecao_entorno = ",\n        ".join(
    f'e."{coluna}"'
    for coluna in novas_colunas
)

# Registrar a consulta de integração
con.execute(f"""
    CREATE OR REPLACE VIEW universo_v2 AS
    SELECT
        b.*,
        {selecao_entorno}
    FROM universo_v1_local AS b
    LEFT JOIN entorno_dados AS e
        ON b.CD_SETOR = e.CD_SETOR
""")

# Exportar primeiro para um arquivo temporário
caminho_temp_sql = str(
    arquivo_v2_temp
).replace("'", "''")

con.execute(f"""
    COPY (
        SELECT *
        FROM universo_v2
    )
    TO '{caminho_temp_sql}'
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD
    )
""")

# Promover o arquivo temporário à versão definitiva
arquivo_v2_temp.replace(arquivo_v2)

print("BANCO NACIONAL V2 CRIADO!")
print("Arquivo:", arquivo_v2)
print(
    "Tamanho (MB):",
    round(arquivo_v2.stat().st_size / 1024**2, 2)
)


# Validação estrutural do banco integrado

São verificados o total de registros, a unicidade da chave territorial, a ausência de identificadores nulos e a quantidade de colunas. A estrutura esperada é de 468.099 setores censitários e 1.391 colunas.

In [ ]:
# 13. VALIDAÇÃO DO BANCO NACIONAL -VERSÃO 2

caminho_v2_sql = str(arquivo_v2).replace("'", "''")

con.execute(f"""
    CREATE OR REPLACE VIEW universo_v2_arquivo AS
    SELECT *
    FROM read_parquet('{caminho_v2_sql}')
""")

validacao_v2 = con.execute("""
    SELECT
        COUNT(*) AS registros,
        COUNT(DISTINCT CD_SETOR) AS setores_unicos,
        COUNT(*) FILTER (
            WHERE CD_SETOR IS NULL
        ) AS codigos_nulos,
        COUNT(ENT_V05000) AS setores_com_entorno
    FROM universo_v2_arquivo
""").fetchdf()

colunas_v2 = con.execute("""
    DESCRIBE SELECT * FROM universo_v2_arquivo
""").fetchdf()

display(validacao_v2)

print("Colunas totais:", len(colunas_v2))
print("Novas variáveis:", len(colunas_v2) - 1356)

# Validações de integridade
assert validacao_v2.loc[0, "registros"] == 468099
assert validacao_v2.loc[0, "setores_unicos"] == 468099
assert validacao_v2.loc[0, "codigos_nulos"] == 0
assert len(colunas_v2) == 1391

# Confirmar todas as variáveis de Entorno
assert all(
    coluna in colunas_v2["column_name"].values
    for coluna in novas_colunas
)

print("\nEstrutura nacional da V2 validada!")

# Alias utilizados pelas auditorias seguintes
con.execute("""
    CREATE OR REPLACE VIEW v1 AS
    SELECT * FROM universo_v1_local
""")

con.execute("""
    CREATE OR REPLACE VIEW v2 AS
    SELECT * FROM universo_v2_arquivo
""")

# Listas das colunas para comparação
colunas_v1 = con.execute(
    "DESCRIBE SELECT * FROM v1"
).fetchdf()["column_name"].tolist()

colunas_v2 = con.execute(
    "DESCRIBE SELECT * FROM v2"
).fetchdf()["column_name"].tolist()

assert colunas_v2[:len(colunas_v1)] == colunas_v1


# Auditoria da preservação das variáveis originais

Esta etapa compara os valores da V1 e da V2 mediante correspondência exata por CD_SETOR. A comparação é dividida em lotes de 20 variáveis para reduzir o consumo de memória. A operação IS NOT DISTINCT FROM permite identificar diferenças entre valores, inclusive quando um deles é nulo. O procedimento abrange as 1.355 colunas de dados originais, excluindo a chave territorial, cuja unicidade foi verificada anteriormente.

In [ ]:
# 16. VALIDAÇÃO INTEGRAL  V1 x V2
# Comparação por lotes de 20 variáveis

import pandas as pd
import time

# Variáveis originais, excluindo a chave territorial
variaveis_validar = [
    c for c in colunas_v1 if c != "CD_SETOR"
]

tamanho_lote = 20
resultados = []

inicio = time.time()

def proteger_coluna(nome):
    return '"' + nome.replace('"', '""') + '"'

for i in range(0, len(variaveis_validar), tamanho_lote):

    lote = variaveis_validar[i:i + tamanho_lote]

    condicoes = " OR ".join(
        f'NOT (a.{proteger_coluna(c)} '
        f'IS NOT DISTINCT FROM b.{proteger_coluna(c)})'
        for c in lote
    )

    divergencias = con.execute(f"""
        SELECT COUNT(*)
        FROM v1 a
        INNER JOIN v2 b
            ON a.CD_SETOR = b.CD_SETOR
        WHERE {condicoes}
    """).fetchone()[0]

    numero_lote = i // tamanho_lote + 1

    resultados.append({
        "Lote": numero_lote,
        "Variáveis": len(lote),
        "Setores com divergência": divergencias
    })

    print(
        f"Lote {numero_lote:02d} | "
        f"{len(lote)} variáveis | "
        f"Divergências: {divergencias}",
        flush=True
    )

# Relatório final
relatorio_validacao = pd.DataFrame(resultados)

total_divergencias = (
    relatorio_validacao["Setores com divergência"].sum()
)

print("\n=== RESULTADO FINAL ===")
print("Lotes verificados:", len(resultados))
print("Variáveis verificadas:", len(variaveis_validar))
print("Ocorrências de divergência por lote:", total_divergencias)
print("Tempo (min):", round((time.time() - inicio) / 60, 2))

assert total_divergencias == 0, (
    "Foram encontradas diferenças entre V1 e V2."
)

print("\nVALIDAÇÃO INTEGRAL CONCLUÍDA!")
print("Todos os valores originais foram preservados.")


# Auditoria das variáveis acrescentadas
As 35 variáveis de Entorno armazenadas na V2 são comparadas com os valores disponíveis no CSV oficial do IBGE. A comparação considera os códigos CD_SETOR e verifica diferenças de valores e de preenchimento. Adicionalmente, são identificados os setores da V2 que não possuem correspondência no levantamento de Entorno e confirmada a presença de valores nulos nas 35 colunas acrescentadas.

In [ ]:
# 17. VALIDAÇÃO DAS VARIÁVEIS DE ENTORNO -IBGE x V2

import pandas as pd
import zipfile
import time

pasta_entorno = pasta_base / "ENTORNO"

arquivo_zip = (
    pasta_entorno /
    "Agregados_por_setores_entorno_domicílios_BR.zip"
)

# Ler o CSV original diretamente do ZIP
with zipfile.ZipFile(arquivo_zip) as z:
    nome_csv = next(
        n for n in z.namelist()
        if n.lower().endswith(".csv")
    )

    with z.open(nome_csv) as f:
        entorno_original = pd.read_csv(
            f,
            sep=";",
            encoding="utf-8-sig",
            dtype=str,
            low_memory=False
        )

entorno_original.rename(
    columns={"CD_setor": "CD_SETOR"},
    inplace=True
)

variaveis = [f"V050{i:02d}" for i in range(35)]

assert all(
    c in entorno_original.columns for c in variaveis
)

assert entorno_original["CD_SETOR"].is_unique
assert entorno_original["CD_SETOR"].notna().all()
assert len(entorno_original) == 340965

# Registrar os dados originais no DuckDB
con.register("entorno_original", entorno_original)

# Comparar em lotes de 10 variáveis
resultados_entorno = []
inicio = time.time()

for i in range(0, 35, 10):
    lote = variaveis[i:i+10]

    condicoes = " OR ".join(
        f'NOT (e."{v}" IS NOT DISTINCT FROM '
        f'b."ENT_{v}")'
        for v in lote
    )

    divergencias = con.execute(f"""
        SELECT COUNT(*)
        FROM entorno_original e
        INNER JOIN v2 b
            ON e.CD_SETOR = b.CD_SETOR
        WHERE {condicoes}
    """).fetchone()[0]

    resultados_entorno.append({
        "Lote": i // 10 + 1,
        "Variáveis": len(lote),
        "Divergências": divergencias
    })

    print(
        f"Lote {i//10 + 1}: "
        f"{len(lote)} variáveis | "
        f"Divergências: {divergencias}",
        flush=True
    )

relatorio_entorno = pd.DataFrame(resultados_entorno)

# Conferir setores sem correspondência no Entorno
# e verificar se as novas colunas estão nulas neles
condicao_nulos = " AND ".join(
    f'b."ENT_{v}" IS NULL' for v in variaveis
)

ausencias = con.execute(f"""
    SELECT
        COUNT(*) AS setores_sem_entorno,
        COUNT(*) FILTER (
            WHERE {condicao_nulos}
        ) AS setores_com_35_valores_nulos
    FROM v2 b
    ANTI JOIN entorno_original e
        ON b.CD_SETOR = e.CD_SETOR
""").fetchdf()

display(relatorio_entorno)
display(ausencias)

assert relatorio_entorno["Divergências"].sum() == 0
assert ausencias.loc[0, "setores_sem_entorno"] == 127134
assert (
    ausencias.loc[0, "setores_com_35_valores_nulos"]
    == 127134
)

print("\nVALIDAÇÃO DO ENTORNO CONCLUÍDA!")
print("35 variáveis verificadas.")
print("Nenhuma divergência encontrada.")
print(
    "Tempo (min):",
    round((time.time() - inicio) / 60, 2)
)


In [ ]:

# 18. ORGANIZAÇÃO DO BANCO V2 -RESULTADOS_2022

from pathlib import Path
import shutil

origem = Path(
    "/content/drive/MyDrive/PRISCILA/"
    "#_DOUTORADO/0_TESE/DADOS/IBGE_UNIVERSO/"
    "UNIVERSO_2022_NACIONAL_V2.parquet"
)

pasta_destino = origem.parent / "RESULTADOS_2022"
destino = pasta_destino / origem.name

pasta_destino.mkdir(parents=True, exist_ok=True)

if not origem.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {origem}")

if destino.exists():
    raise FileExistsError(
        f"Já existe um arquivo no destino: {destino}"
    )

# Copiar sem apagar a origem
shutil.copy2(origem, destino)

# Verificar tamanho da cópia
assert destino.stat().st_size == origem.stat().st_size

print("V2 copiada para RESULTADOS_2022!")
print("Localização:", destino)
print(
    "Tamanho (MB):",
    round(destino.stat().st_size / 1024**2, 2)
)


# 19. RELATÓRIO TÉCNICO E AUDITORIA - UNIVERSO 2022 V2

Objetivo:
Consolidar os resultados das auditorias realizadas durante a construção do banco nacional V2 e gerar documentação técnica para fins de rastreabilidade e reprodutibilidade.

Entradas:
UNIVERSO_2022_NACIONAL.parquet (V1)
UNIVERSO_2022_NACIONAL_V2.parquet (V2)
relatorio_validacao (auditoria das 1.355 variáveis)
relatorio_entorno (auditoria das 35 variáveis)
ausencias (cobertura do conjunto Entorno)

Saídas:
RELATORIO_VALIDACAO_UNIVERSO_2022_V2.xlsx
RELATORIO_TECNICO_UNIVERSO_2022_V2.md

In [ ]:

# 19. RELATÓRIO TÉCNICO E AUDITORIA — UNIVERSO 2022 V2


import pandas as pd
import duckdb
import pyarrow
import platform
from datetime import datetime

# ---------------------------------------------------
# 1. DEFINIR OS ARQUIVOS DE ENTRADA E SAÍDA
# ---------------------------------------------------

# Diretórios definidos na configuração inicial
pasta_base = PASTA_BASE
pasta_resultados = PASTA_RESULTADOS

arquivo_v1 = (
    pasta_base / "UNIVERSO_2022_NACIONAL.parquet"
)

arquivo_v2 = (
    pasta_resultados /
    "UNIVERSO_2022_NACIONAL_V2.parquet"
)

saida_excel = (
    pasta_resultados /
    "RELATORIO_VALIDACAO_UNIVERSO_2022_V2.xlsx"
)

saida_md = (
    pasta_resultados /
    "RELATORIO_TECNICO_UNIVERSO_2022_V2.md"
)

# Verificar a disponibilidade dos bancos
assert arquivo_v1.is_file(), "Arquivo V1 não encontrado."
assert arquivo_v2.is_file(), "Arquivo V2 não encontrado."

# Proteger os relatórios já existentes
if (
    saida_excel.exists() or saida_md.exists()
) and not SOBRESCREVER_SAIDAS:
    raise FileExistsError(
        "Os relatórios da V2 já existem. "
        "A sobrescrita está desabilitada."
    )

# ---------------------------------------------------
# 2. VERIFICAR OS RESULTADOS DAS AUDITORIAS
# ---------------------------------------------------

for nome in [
    "relatorio_validacao",
    "relatorio_entorno",
    "ausencias"
]:
    if nome not in globals():
        raise RuntimeError(
            f"Resultado {nome} não encontrado. "
            "Execute as células de auditoria antes."
        )

# Validação das variáveis originais
assert len(relatorio_validacao) == 68

assert (
    relatorio_validacao["Variáveis"].sum() == 1355
)

assert (
    relatorio_validacao[
        "Setores com divergência"
    ].sum() == 0
)

# Validação das variáveis de Entorno
assert len(relatorio_entorno) == 4

assert (
    relatorio_entorno["Variáveis"].sum() == 35
)

assert (
    relatorio_entorno["Divergências"].sum() == 0
)

# Validação da cobertura territorial
assert int(
    ausencias.loc[0, "setores_sem_entorno"]
) == 127134

assert int(
    ausencias.loc[0, "setores_com_35_valores_nulos"]
) == 127134

print("Resultados das auditorias conferidos.")

# ---------------------------------------------------
# 3. REGISTRAR AS VERSÕES DO AMBIENTE
# ---------------------------------------------------

versao_python = platform.python_version()
versao_duckdb = duckdb.__version__
versao_pandas = pd.__version__
versao_pyarrow = pyarrow.__version__

data = datetime.now().strftime("%d/%m/%Y")

print("\n=== AMBIENTE DE PROCESSAMENTO ===")
print("Python:", versao_python)
print("DuckDB:", versao_duckdb)
print("Pandas:", versao_pandas)
print("PyArrow:", versao_pyarrow)

# ---------------------------------------------------
# 4. CONSOLIDAR O RESUMO DA VALIDAÇÃO
# ---------------------------------------------------

resumo_validacao = pd.DataFrame([
    ["Fonte", "Censo Demográfico 2022 — IBGE"],
    ["Abrangência", "Brasil"],
    ["Unidade territorial", "Setor censitário"],
    ["Chave de integração", "CD_SETOR"],
    ["Procedimento", "LEFT JOIN"],
    ["Registros V1", 468099],
    ["Registros V2", 468099],
    ["Colunas V1", 1356],
    ["Colunas V2", 1391],
    ["Variáveis incorporadas", 35],
    ["Setores com correspondência ENT", 340965],
    ["Setores sem correspondência ENT", 127134],
    ["Variáveis originais comparadas", 1355],
    ["Divergências V1 x V2", 0],
    ["Divergências Entorno x V2", 0],
    ["Validação", "Aprovada"],
    ["Data da execução", data],
    ["Python", versao_python],
    ["DuckDB", versao_duckdb],
    ["Pandas", versao_pandas],
    ["PyArrow", versao_pyarrow]
], columns=["Indicador", "Resultado"])

# ---------------------------------------------------
# 5. EXPORTAR RELATÓRIO DE VALIDAÇÃO EM EXCEL
# ---------------------------------------------------

with pd.ExcelWriter(
    saida_excel,
    engine="xlsxwriter"
) as writer:

    resumo_validacao.to_excel(
        writer,
        sheet_name="Resumo",
        index=False
    )

    relatorio_validacao.to_excel(
        writer,
        sheet_name="Auditoria V1 V2",
        index=False
    )

    relatorio_entorno.to_excel(
        writer,
        sheet_name="Auditoria Entorno",
        index=False
    )

    ausencias.to_excel(
        writer,
        sheet_name="Cobertura Entorno",
        index=False
    )

    workbook = writer.book

    formato_cabecalho = workbook.add_format({
        "bold": True,
        "font_color": "white",
        "bg_color": "#243746",
        "valign": "vcenter",
        "text_wrap": True
    })

    for aba in writer.sheets.values():
        aba.freeze_panes(1, 0)
        aba.set_row(0, 30, formato_cabecalho)
        aba.set_column(0, 0, 35)
        aba.set_column(1, 4, 28)

# ---------------------------------------------------
# 6. GERAR RELATÓRIO TÉCNICO EM MARKDOWN
# ---------------------------------------------------

texto = f"""# Relatório técnico de construção e validação

## Banco Nacional de Agregados por Setores Censitários — Censo 2022 (V2)

**Data de geração:** {data}

### 1. Objetivo

Documentar a incorporação dos agregados de Características
Urbanísticas do Entorno dos Domicílios à base nacional
integrada por setores censitários do Censo Demográfico 2022.

### 2. Fontes de dados

- Instituto Brasileiro de Geografia e Estatística (IBGE).
- Censo Demográfico 2022 — Agregados por Setores Censitários.
- Banco Nacional Integrado V1.
- Características Urbanísticas do Entorno dos Domicílios.
- Dicionário oficial do Entorno dos Domicílios.

A versão original do banco está disponível em:

https://doi.org/10.5281/zenodo.23599214

### 3. Procedimento de integração

A integração foi realizada por correspondência exata
entre os códigos dos setores censitários (`CD_SETOR`).

Foi utilizada a operação `LEFT JOIN`, preservando
todos os registros existentes na base nacional V1.

As 35 variáveis de Entorno receberam o prefixo `ENT_`,
com preservação dos códigos originais do IBGE.

Os valores originais foram mantidos como texto.

### 4. Estrutura do banco integrado

- Setores censitários da V1: 468.099.
- Setores censitários da V2: 468.099.
- Colunas da V1: 1.356.
- Colunas da V2: 1.391.
- Variáveis de Entorno incorporadas: 35.
- Setores com correspondência no Entorno: 340.965.
- Setores sem correspondência no Entorno: 127.134.

### 5. Validação de integridade

Foram comparadas 1.355 variáveis originais
entre as versões V1 e V2, mediante correspondência
territorial por `CD_SETOR`.

A comparação foi realizada em 68 lotes,
com até 20 variáveis por lote.

**Resultado: nenhuma divergência identificada.**

As 35 variáveis de Entorno foram comparadas com
o arquivo CSV original do IBGE em quatro lotes,
com até 10 variáveis por lote.

**Resultado: nenhuma divergência identificada.**

Também foi confirmada a presença de valores nulos
nas 35 variáveis de Entorno para os 127.134 setores
sem correspondência com o arquivo original.

### 6. Limitações e considerações metodológicas

Embora o banco integrado tenha abrangência nacional,
os agregados de Entorno não apresentam registros
correspondentes para todos os setores da base V1.

A ausência de correspondência não deve ser
interpretada como ausência de infraestrutura urbana.

A interpretação das variáveis exige a consulta
aos dicionários e às notas metodológicas do IBGE.

As verificações realizadas demonstram a integridade
do processo de integração em relação aos arquivos
utilizados. Não representam validação independente
das estatísticas originalmente produzidas pelo IBGE.

### 7. Arquivos produzidos

- `UNIVERSO_2022_NACIONAL_V2.parquet`
- `DICIONARIO_COMPLETO_2022_V2.csv`
- `DICIONARIO_COMPLETO_2022_V2.xlsx`
- `RELATORIO_VALIDACAO_UNIVERSO_2022_V2.xlsx`
- `RELATORIO_TECNICO_UNIVERSO_2022_V2.md`

### 8. Ambiente de processamento

- Python: {versao_python}
- DuckDB: {versao_duckdb}
- Pandas: {versao_pandas}
- PyArrow: {versao_pyarrow}

### 9. Situação da validação

**Integração e verificações de integridade aprovadas.**

A análise substantiva das variáveis e a avaliação
de sua adequação a aplicações específicas constituem
etapas posteriores à validação do banco.
"""

saida_md.write_text(
    texto,
    encoding="utf-8"
)

# ---------------------------------------------------
# 7. APRESENTAR OS RESULTADOS
# ---------------------------------------------------

print("\nRELATÓRIOS GERADOS COM SUCESSO!")

print("\nRelatório Excel:")
print(saida_excel)

print("\nRelatório técnico Markdown:")
print(saida_md)

display(resumo_validacao)


In [ ]:

# 18. DICIONÁRIO CONSOLIDADO — UNIVERSO 2022 V2
#
# Objetivo:
# Incorporar as descrições oficiais das 35 variáveis de
# Entorno ao dicionário original da V1.
#
# Entradas:
# - DICIONARIO_COMPLETO_2022.csv (V1)
# - dicionarios_de_dados_entorno.zip (IBGE)
# - UNIVERSO_2022_NACIONAL_V2.parquet
#
# Saídas:
# - DICIONARIO_COMPLETO_2022_V2.csv
# - DICIONARIO_COMPLETO_2022_V2.xlsx

import pandas as pd
import zipfile
import io
import pyarrow.parquet as pq

# --------------------------------------------------
# 1. DEFINIR OS ARQUIVOS DE ENTRADA
# --------------------------------------------------

# Os diretórios foram definidos na célula inicial
pasta_dicionario = PASTA_DICIONARIOS
pasta_entorno = PASTA_ENTORNO
pasta_resultados = PASTA_RESULTADOS

arquivo_dic_v1 = (
    pasta_dicionario / "DICIONARIO_COMPLETO_2022.csv"
)

arquivo_zip = (
    pasta_entorno / "dicionarios_de_dados_entorno.zip"
)

arquivo_v2 = (
    pasta_resultados /
    "UNIVERSO_2022_NACIONAL_V2.parquet"
)

# Verificar se os arquivos necessários existem
for arquivo in [arquivo_dic_v1, arquivo_zip, arquivo_v2]:
    assert arquivo.is_file(), (
        f"Arquivo de entrada não encontrado: {arquivo}"
    )

# --------------------------------------------------
# 2. CARREGAR O DICIONÁRIO ORIGINAL DA V1
# --------------------------------------------------

dic_v1 = pd.read_csv(
    arquivo_dic_v1,
    dtype=str,
    keep_default_na=False
)

assert len(dic_v1) == 1356, (
    "O dicionário original deve conter 1.356 registros."
)

assert dic_v1["coluna_banco"].is_unique, (
    "Existem identificadores duplicados no dicionário V1."
)

# --------------------------------------------------
# 3. LER O DICIONÁRIO OFICIAL DE ENTORNO
# --------------------------------------------------

with zipfile.ZipFile(arquivo_zip) as z:
    nome = next(
        n for n in z.namelist()
        if n.endswith("dicionario_entorno_domicilios.xlsx")
    )

    with z.open(nome) as f:
        conteudo = f.read()

dic_entorno = pd.read_excel(
    io.BytesIO(conteudo),
    sheet_name="dicionario_entorno_domicilios_v",
    dtype=str
)

dic_entorno.columns = ["Variável", "Descrição"]

dic_entorno["Variável"] = (
    dic_entorno["Variável"].str.strip()
)

assert len(dic_entorno) == 35, (
    "O dicionário de Entorno deve conter 35 variáveis."
)

assert dic_entorno["Variável"].is_unique, (
    "Existem códigos duplicados no dicionário de Entorno."
)

# --------------------------------------------------
# 4. PREPARAR OS REGISTROS DAS 35 NOVAS VARIÁVEIS
# --------------------------------------------------

registros = []

for _, linha in dic_entorno.iterrows():

    codigo = linha["Variável"]

    registro = {
        campo: ""
        for campo in dic_v1.columns
    }

    registro.update({
        "coluna_banco": f"ENT_{codigo}",
        "conjunto": "ENT",
        "tema_ibge": (
            "Características Urbanísticas do "
            "Entorno dos Domicílios"
        ),
        "codigo_original": codigo,
        "descricao_ibge": linha["Descrição"],
        "situacao_documentacao": (
            "Descrição oficial identificada"
        ),
        "fonte_descricao": (
            "IBGE — Censo Demográfico 2022, "
            "Dicionário Entorno Domicílios"
        )
    })

    registros.append(registro)

dic_novas = pd.DataFrame(registros)

# --------------------------------------------------
# 5. CONSOLIDAR OS DICIONÁRIOS V1 E ENTORNO
# --------------------------------------------------

dicionario_v2 = pd.concat(
    [dic_v1, dic_novas],
    ignore_index=True
)

# Recuperar a estrutura efetiva do Parquet V2
colunas_parquet = (
    pq.ParquetFile(arquivo_v2).schema_arrow.names
)

# Verificar quantidade, unicidade e ordem das colunas
assert len(colunas_parquet) == 1391

assert (
    dicionario_v2["coluna_banco"].tolist()
    == colunas_parquet
), (
    "A ordem dos registros do dicionário "
    "não corresponde às colunas do Parquet V2."
)

assert dicionario_v2["coluna_banco"].is_unique

# Registrar a posição de cada coluna
dicionario_v2["ordem"] = range(1, 1392)

# --------------------------------------------------
# 6. GERAR O RESUMO POR CONJUNTO TEMÁTICO
# --------------------------------------------------

resumo = (
    dicionario_v2.groupby(
        ["conjunto", "tema_ibge"],
        as_index=False
    )
    .agg(
        total_variaveis=("coluna_banco", "count")
    )
)

# --------------------------------------------------
# 7. DEFINIR OS ARQUIVOS DE SAÍDA
# --------------------------------------------------

saida_csv = (
    pasta_dicionario /
    "DICIONARIO_COMPLETO_2022_V2.csv"
)

saida_xlsx = (
    pasta_dicionario /
    "DICIONARIO_COMPLETO_2022_V2.xlsx"
)

# Evitar sobrescrever arquivos existentes
if (
    saida_csv.exists() or saida_xlsx.exists()
) and not SOBRESCREVER_SAIDAS:
    raise FileExistsError(
        "Os dicionários V2 já existem. "
        "A sobrescrita está desabilitada."
    )

# --------------------------------------------------
# 8. EXPORTAR O DICIONÁRIO EM CSV
# --------------------------------------------------

dicionario_v2.to_csv(
    saida_csv,
    index=False,
    encoding="utf-8-sig"
)

# --------------------------------------------------
# 9. EXPORTAR O DICIONÁRIO EM EXCEL
# --------------------------------------------------

with pd.ExcelWriter(
    saida_xlsx,
    engine="xlsxwriter"
) as writer:

    dicionario_v2.to_excel(
        writer,
        sheet_name="Dicionario V2",
        index=False
    )

    dic_novas.to_excel(
        writer,
        sheet_name="Novas variaveis ENT",
        index=False
    )

    resumo.to_excel(
        writer,
        sheet_name="Resumo",
        index=False
    )

    workbook = writer.book

    formato_cabecalho = workbook.add_format({
        "bold": True,
        "bg_color": "#243746",
        "font_color": "white",
        "text_wrap": True,
        "valign": "vcenter"
    })

    for aba in writer.sheets.values():
        aba.freeze_panes(1, 0)
        aba.set_row(0, 32, formato_cabecalho)
        aba.set_column(0, 0, 12)
        aba.set_column(1, 3, 28)
        aba.set_column(4, 10, 45)

# --------------------------------------------------
# 10. APRESENTAR OS RESULTADOS
# --------------------------------------------------

print("DICIONÁRIO V2 CRIADO!")
print("Total de colunas:", len(dicionario_v2))
print("Novas variáveis:", len(dic_novas))

print("\nArquivos de saída:")
print(saida_csv)
print(saida_xlsx)

display(resumo)
